# 02b. Walk Through `sampling_generation.py`

This notebook expands the important parts of `src/apps/sampling/sampling_generation.py` into separate cells. The setup and IO helpers are still imported from the repo, but the generation loop, masks, metrics, control-cell merge, and AnnData construction are shown explicitly so you can inspect what is happening inside the function.

The flow mirrors `rawdata_diffusion_sampling.py`:

1. Compose the Hydra config used by the sampling command.
2. Build the datamodule and populate covariate cardinalities/dictionaries.
3. Load the checkpoint-backed model.
4. Create the sampling dataloader and inspect one batch.
5. Run one DDIM/DDPM batch and inspect masks, selected genes, metrics, and covariates.
6. Build the control-cell tail and AnnData objects that would be saved.


## 0. Imports and Paths

Keep this notebook in the repository root context. If your notebook server starts elsewhere, set `PROJECT_ROOT` manually.

In [1]:
from pathlib import Path
import sys
import time
import gc

PROJECT_ROOT = Path("/home/lwf/PerturbDiff").resolve()
assert (PROJECT_ROOT / "src" / "apps" / "run" / "rawdata_diffusion_sampling.py").exists(), PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "replogle"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME
PERTURB_ROOT = DATA_ROOT / "perturb_data"
CKPT_PATH = PROJECT_ROOT / "checkpoints" / DATASET_NAME / "from_scratch_replogle.ckpt"
RUN_DIR = PROJECT_ROOT / "runs" / "replogle_sampling_walkthrough"
WANDB_DIR = PROJECT_ROOT / "wandb"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PERTURB_ROOT:", PERTURB_ROOT)
print("CKPT_PATH:", CKPT_PATH)
assert PERTURB_ROOT.exists(), "Run 00_download_data.ipynb first."
assert CKPT_PATH.exists(), "Run 00_download_data.ipynb first."

PROJECT_ROOT: /home/lwf/PerturbDiff
PERTURB_ROOT: /home/lwf/PerturbDiff/data/replogle/perturb_data
CKPT_PATH: /home/lwf/PerturbDiff/checkpoints/replogle/from_scratch_replogle.ckpt


In [2]:
import anndata
import numpy as np
import torch
from geomloss import SamplesLoss
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf
from pytorch_lightning.utilities import move_data_to_device
from sklearn.metrics import r2_score

from src.common.utils import setup_loggings
from src.apps.sampling.sampling_setup import (
    build_sampling_datamodule,
    load_sampling_model,
    populate_covariate_cfg,
)
from src.apps.sampling.sampling_utils import setup_device
from src.apps.sampling.sampling_generation_helpers import (
    build_gene_embedding_cache,
    build_obs_data,
    build_self_condition,
    build_x_ctrl,
    collect_batch_covariates,
    load_ctrl_adata,
    load_selected_genes,
    resolve_sampling_runner,
)
from src.apps.sampling.sampling_io import save_adata

## 1. Compose the Same Hydra Config

This is the notebook equivalent of the command-line overrides in `02_inference_sampling.ipynb`. The important point: this produces the `cfg` that is passed into `generate_samples()`.

In [3]:
overrides = [
    "path=trixie_path",
    f"path.tmp_dir={PERTURB_ROOT}",
    f"path.diffusion.save_dir={RUN_DIR}",
    f"path.wandb.logging_dir={WANDB_DIR}",
    f"model_checkpoint_path={CKPT_PATH}",
    "data=replogle_finetune",
    "data.normalize_counts=10",
    "data.pad_length=2000",
    "data.embed_key=X_hvg",
    "model.hidden_num=[2000,512]",
    "model.input_dim=2000",
    "optimization.micro_batch_size=16",
    "data.use_cell_set=8",
    "cov_encoding=trixie_onehot",
    "cov_encoding.batch_encoding=onehot",
    "cov_encoding.celltype_encoding=llm",
    "cov_encoding.replogle_gene_encoding=genept",
    "model.p_drop_control=0",
    "data.keep_control_cell=false",
    "sampling.use_ddim=true",
    "sampling.num_sampled_batches=1",
    "trainer.devices=[0]",
    "trainer.use_distributed_sampler=false",
    "lightning.logger._target_=pytorch_lightning.loggers.logger.DummyLogger",
    "~lightning.logger.project",
    "~lightning.logger.save_dir",
    "~lightning.logger.name",
]

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "configs"), version_base=None):
    cfg = compose(config_name="rawdata_diffusion_sampling", overrides=overrides)

OmegaConf.resolve(cfg)
logger = setup_loggings(cfg)

print("save_dir_path:", cfg.save_dir_path)
print("dataset_path:", cfg.data.dataset_path)
print("selected_gene_file:", cfg.data.selected_gene_file)
print("replogle_gene_embedding_path:", cfg.cov_encoding.replogle_gene_embedding_path)
print("output_dir:", cfg.sampling.output_dir)

05/19/2026 15:48:56 - INFO - src.common.utils -  Configuration args
05/19/2026 15:48:56 - INFO - src.common.utils -  {'run_name': 'sampling', 'save_dir_path': None, 'model_checkpoint_path': '/home/lwf/PerturbDiff/checkpoints/replogle/from_scratch_replogle.ckpt', 'device': 'auto', 'sampling': {'sample_unperturbed': False, 'num_sampled_batches': 1, 'batch_size': 16, 'use_ddim': True, 'clip_denoised': True, 'progress': True, 'guidance_strength': 1.0, 'start_time': 100, 'nw': 0.5, 'start_guide_steps': 500, 'eta': 0.0, 'output_format': 'npz', 'output_dir': None}, 'trainer': {'accelerator': 'gpu', 'devices': [0], 'num_nodes': 1, 'use_distributed_sampler': False, 'default_root_dir': '/home/lwf/PerturbDiff/runs/replogle_sampling_walkthrough', 'enable_checkpointing': True, 'enable_model_summary': True, 'enable_progress_bar': True, 'deterministic': True, 'check_val_every_n_epoch': 1, 'val_check_interval': 1.0, 'log_every_n_steps': 20, 'num_sanity_val_steps': 0, 'max_epochs': None, 'max_steps': 1

## 2. Build Datamodule and Model

This is still setup code, so we call the existing helpers. After `populate_covariate_cfg`, `cfg.cov_encoding` contains the runtime dictionaries that the model's covariate encoder needs.

In [4]:
torch.manual_seed(cfg.optimization.seed)

t0 = time.time()
datamodule = build_sampling_datamodule(cfg, logger)
populate_covariate_cfg(cfg, datamodule)
print(f"datamodule setup took {time.time() - t0:.2f}s")
print("num_pert:", cfg.cov_encoding.num_pert)
print("num_celltype:", cfg.cov_encoding.num_celltype)
print("num_batch:", cfg.cov_encoding.num_batch)
print("all_split_names:", datamodule.all_split_names)

05/19/2026 15:51:26 - INFO - src.common.utils -  Calling pretraining data module setup ...


Processing meta data ...: 1it [00:00,  1.75it/s]

05/19/2026 15:51:27 - INFO - src.common.utils -  #perturbation: 2024, #cell type: 4, #batch: 56
datamodule setup took 0.68s
num_pert: 2024
num_celltype: 4
num_batch: 56
all_split_names: ['validation', 'test']


In [5]:
t0 = time.time()
model = load_sampling_model(cfg, logger, datamodule)
device = setup_device(cfg, logger)
model = model.to(device)
model.eval()
print(f"model load took {time.time() - t0:.2f}s")
print("device:", device)
print("model type:", model.model_cfg.model_type)
print("diffusion steps:", model.model_cfg.steps)

05/19/2026 15:51:40 - INFO - src.common.utils -  Using device: cuda
model load took 6.70s
device: cuda
model type: Cross_DiT
diffusion steps: 1000


## 3. Build the Sampling Dataloader

`generate_samples()` uses `datamodule.val_dataloader()[1]`, which is the test split dataloader in this project. This setup can be slow because it constructs split indices, group caches, and sampler state.

In [ ]:
t0 = time.time()
datamodule.setup_dataset()
val_loaders = datamodule.val_dataloader()
dataloader = val_loaders[1]
dataloader_iter = iter(dataloader)
print(f"dataset/dataloader setup took {time.time() - t0:.2f}s")
print("number of validation loaders:", len(val_loaders))
print("test sampler length:", len(dataloader.sampler))
print("dataset length:", len(dataloader.dataset))

## 4. Resolve Sampling Parameters

This mirrors the top part of `generate_samples()`. With `optimization.micro_batch_size=16` and `data.use_cell_set=8`, the function samples a dataloader batch size of `16 // 8 = 2` cell sets.

In [ ]:
batch_size = cfg.sampling.batch_size
if cfg.data.use_cell_set is not None:
    batch_size = int(batch_size // cfg.data.use_cell_set)

num_sampled_batches = cfg.sampling.num_sampled_batches
if num_sampled_batches is None:
    num_sampled_batches = len(dataloader)

num_samples = len(dataloader.sampler)
cell_set_number = cfg.data.use_cell_set if cfg.data.use_cell_set is not None else 1
input_dim = getattr(cfg.model, "input_dim", None) or getattr(cfg.model, "output_size", None) or getattr(cfg.model, "gene_dim", None)
normalize_counts = getattr(getattr(dataloader.dataset, "data_args", None), "normalize_counts", None)
selected_genes = load_selected_genes(cfg)

print("batch_size used by sampling_generation:", batch_size)
print("cell_set_number:", cell_set_number)
print("num_sampled_batches:", num_sampled_batches)
print("num_samples reported by sampler:", num_samples)
print("input_dim:", input_dim)
print("normalize_counts:", normalize_counts)
print("selected_genes:", len(selected_genes), selected_genes[:5])

## 5. Pull One Batch and Inspect It

The batch already contains the target perturbed expression (`pert_emb`), control expression/conditioning fields, covariate ids, selected column gene names, and padding markers.

In [ ]:
def describe_value(x):
    if torch.is_tensor(x):
        return f"Tensor shape={tuple(x.shape)} dtype={x.dtype} device={x.device}"
    if isinstance(x, np.ndarray):
        return f"ndarray shape={x.shape} dtype={x.dtype}"
    if isinstance(x, (list, tuple)):
        head = type(x[0]).__name__ if len(x) else "empty"
        return f"{type(x).__name__} len={len(x)} first_type={head}"
    return f"{type(x).__name__}: {str(x)[:120]}"

batch_data_cpu = next(dataloader_iter)
print("batch keys:")
for key in sorted(batch_data_cpu.keys()):
    print(f"  {key}: {describe_value(batch_data_cpu[key])}")

## 6. Move Batch to GPU and Build Conditioning

This is the core pre-sampling preparation:

- `model._encode_covariates(batch_data)` converts perturbation/cell type/batch ids into a conditioning vector.
- `pert_emb` is the real target expression for this batch.
- `is_padded_list` marks duplicate/padded entries introduced by the sampler to make cell sets uniform.
- `self_condition` is the dictionary passed to the diffusion model at every denoising step.

In [ ]:
batch_data = move_data_to_device(batch_data_cpu, device)

with torch.no_grad():
    batch_data["batch_emb"] = model._encode_covariates(batch_data)
    pert_emb = batch_data["pert_emb"]
    resolved_input_dim = int(input_dim) if input_dim is not None else int(pert_emb.shape[-1])
    gene_emb = build_gene_embedding_cache(model, batch_data, device)
    self_condition = build_self_condition(cfg, model, batch_data, gene_emb)
    valid_mask = ~batch_data["is_padded_list"].bool()

print("batch_emb:", describe_value(batch_data["batch_emb"]))
print("pert_emb:", describe_value(pert_emb))
print("gene_emb:", describe_value(gene_emb))
print("valid_mask:", describe_value(valid_mask), "valid entries:", int(valid_mask.sum().item()))
print("resolved_input_dim:", resolved_input_dim)
print("self_condition keys:", sorted(self_condition.keys()))
for key, value in self_condition.items():
    print(f"  {key}: {describe_value(value)}")

## 7. Run One DDIM/DDPM Batch

This is the expensive part of `generate_samples()`. `resolve_sampling_runner()` chooses `diffusion.ddim_sample_loop` when `sampling.use_ddim=true`, otherwise `diffusion.p_sample_loop`.

Set `RUN_DDIM_BATCH = False` if you only want to inspect setup without sampling.

In [ ]:
RUN_DDIM_BATCH = True

sample = None
traj = None
sample_fn, sampling_kwargs = resolve_sampling_runner(cfg, model.diffusion, cfg.sampling.use_ddim)
current_batch_size = len(pert_emb)
sample_shape = (current_batch_size, cell_set_number, resolved_input_dim)

print("sample_fn:", sample_fn)
print("sample_shape:", sample_shape)
print("sampling_kwargs:", sampling_kwargs)

if RUN_DDIM_BATCH:
    t0 = time.time()
    with torch.no_grad():
        sample, traj = sample_fn(
            model.model,
            sample_shape,
            self_condition=self_condition,
            clip_denoised=cfg.sampling.clip_denoised,
            device=device,
            progress=cfg.sampling.progress,
            **sampling_kwargs,
        )
    print(f"sampling took {time.time() - t0:.2f}s")
    print("sample:", describe_value(sample))
    print("traj:", describe_value(traj))

## 8. Remove Padding, Select Genes, and Compute Metrics

The sampler may duplicate entries for padding. The real generated rows are selected with `valid_mask`. Then both truth and sample are restricted/reordered to the 2,000 selected genes before R2 and MMD are computed.

In [ ]:
if sample is None:
    raise RuntimeError("Run the previous cell with RUN_DDIM_BATCH = True first.")

mmd_loss = SamplesLoss(loss="energy", blur=0.05, scaling=0.5).to(device)

pert_valid = pert_emb[valid_mask]
sample_valid = sample[valid_mask]

np_mask = np.isin(batch_data["col_genes"][0], selected_genes)
torch_gene_mask = torch.as_tensor(np_mask, device=device, dtype=torch.bool)

truth_np = pert_valid.detach().cpu().numpy()[:, np_mask]
sample_np = sample_valid.detach().cpu().numpy()[:, np_mask]

r2_metric = r2_score(truth_np.mean(0), sample_np.mean(0))
mmd_metric = mmd_loss(sample_valid[:, torch_gene_mask], pert_valid[:, torch_gene_mask]).item()

print("pert_valid:", describe_value(pert_valid))
print("sample_valid:", describe_value(sample_valid))
print("selected gene mask count:", int(np_mask.sum()))
print("truth_np:", truth_np.shape, truth_np.dtype)
print("sample_np:", sample_np.shape, sample_np.dtype)
print("r2_metric:", r2_metric)
print("mmd_metric:", mmd_metric)

## 9. Decode Batch Covariates Back to Labels

`collect_batch_covariates()` maps integer ids back to perturbation, cell type, and batch labels. These labels become `obs` rows for the output AnnData.

In [ ]:
batch_covariates = collect_batch_covariates(batch_data, dataloader, datamodule, valid_mask)
all_pert = np.concatenate([x[0] for x in batch_covariates], axis=0)
all_celltype = np.concatenate([x[1] for x in batch_covariates], axis=0)
all_batch = np.concatenate([x[2] for x in batch_covariates], axis=0)

print("all_pert:", all_pert.shape, all_pert[:5])
print("all_celltype:", all_celltype.shape, all_celltype[:5])
print("all_batch:", all_batch.shape, all_batch[:5])

## 10. Control Cells and `X_ctrl`

This is the part that happens after `Overall r2_metric` in the original function. For Replogle, it reads `replogle.h5ad`, filters to `cell_line == "hepg2"` and `gene == "non-targeting"`, then selects the same 2,000 genes and converts that slice to a dense array.

In [ ]:
t0 = time.time()
ctrl_adata, var_index = load_ctrl_adata(cfg)
print(f"load_ctrl_adata took {time.time() - t0:.2f}s")
print("ctrl_adata shape:", ctrl_adata.shape)
print("ctrl_adata.X type:", type(ctrl_adata.X))
print("var_index returned by helper:", var_index)

t0 = time.time()
X_ctrl = build_x_ctrl(ctrl_adata, selected_genes, cfg)
print(f"build_x_ctrl took {time.time() - t0:.2f}s")
print("X_ctrl:", X_ctrl.shape, X_ctrl.dtype)
var_index = selected_genes

## 11. Build Output AnnData Objects

The output rows are `[generated rows, control rows]`. The same control rows are appended to both the predicted and true AnnData objects.

In [ ]:
all_samples = sample_np.copy()
all_truths = truth_np.copy()

if normalize_counts is not None and normalize_counts is not False:
    all_samples *= normalize_counts
    all_truths *= normalize_counts

obs = build_obs_data(cfg, all_pert, all_celltype, all_batch, ctrl_adata)
pred_adata = anndata.AnnData(X=np.concatenate([all_samples, X_ctrl]), obs=obs)
true_adata = anndata.AnnData(X=np.concatenate([all_truths, X_ctrl]), obs=obs)
pred_adata.var.index = var_index
true_adata.var.index = var_index

print("obs shape:", obs.shape)
print(obs.head())
print("pred_adata:", pred_adata)
print("true_adata:", true_adata)

## 12. Optional Save

The original function always calls `save_adata()`. Here it is behind a flag so you can inspect objects first.

In [ ]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    save_adata(pred_adata, true_adata, cfg, logger, timestamp=timestamp)
else:
    output_dir = Path(cfg.save_dir_path) / "samples" if cfg.sampling.output_dir is None else Path(cfg.sampling.output_dir)
    print("SAVE_OUTPUTS is False. Would write to:", output_dir)

gc.collect()

## 13. Optional: Compare with the Original Function

After you understand the pieces above, this calls the original `generate_samples()` end to end. It will repeat dataset setup/sampling and save outputs.

In [ ]:
RUN_ORIGINAL_GENERATE_SAMPLES = False

if RUN_ORIGINAL_GENERATE_SAMPLES:
    from src.apps.sampling.sampling_generation import generate_samples
    truths, samples, trajectories, decoded = generate_samples(
        model,
        model.diffusion,
        cfg,
        device,
        logger,
        datamodule,
        pca_for_decode=None,
    )
    print("truths:", truths.shape)
    print("samples:", samples.shape)
    print("trajectories:", type(trajectories), len(trajectories))
    print("decoded:", decoded)